# Прогнозированием с использованием STL-разложения на тренд и сезонность (пакет [`sktime`](https://www.sktime.net/en/stable/))

In [ ]:
import numpy as np
import pandas as pd

from sktime.forecasting.trend import STLForecaster
# модель для прогнозирования компоненты ряда
from sktime.forecasting.arima import StatsModelsARIMA as ARIMA

from sktime.utils.plotting import plot_series

import pandas_datareader.data as web

# настройки визуализации
import matplotlib.pyplot as plt

# Не показывать Warnings
import warnings
warnings.simplefilter(action='ignore', category=Warning)
# Не показывать ValueWarning, ConvergenceWarning из statsmodels
from statsmodels.tools.sm_exceptions import ValueWarning, ConvergenceWarning
warnings.simplefilter('ignore', category=ValueWarning)
warnings.simplefilter('ignore', category=ConvergenceWarning)

Загрузим из БД [`FRED`](https://fred.stlouisfed.org/) дневные данные по Gross Domestic Product () (Symbol [`NA000334Q`](https://fred.stlouisfed.org/series/NA000334Q)) с 1990-01-01 по 2025-12-31 (ряд `z`) и создадим датафрейм `y=log(z)`

In [ ]:
z = web.DataReader(name='NA000334Q', data_source='fred', start='1990-01-01', end='2025-12-31')
z.index = z.index.to_period(freq='Q')
y = np.log(z)

In [ ]:
forecaster = STLForecaster(sp=4, forecaster_trend=ARIMA(order=(1,1,1), trend='t'), 
							forecaster_seasonal=ARIMA(order=(1,0,1), seasonal_order=(1,0,1,4), trend='n'),
							forecaster_resid=None)
y_pred = forecaster.fit_predict(y=y, fh=np.arange(1,11))

In [ ]:
plot_series(y.tail(50), y_pred)
plt.show()